### Fairness Metrics

In [ ]:
df = df_cleaned.copy()

protected_attributes = [
    'Sex_int', 'Protected category', 'Age Range_int',
    'Italian Residence', 'European Residence'
]

df = df.dropna(subset=protected_attributes).reset_index(drop=True)

X = df.drop(columns=['Hired'])[feature_sets['custom_scores_with_essential_base_attributes']]
y = df['Hired']

bool_cols = X.select_dtypes(include='bool').columns
non_bool_cols = X.columns.difference(bool_cols)

X[bool_cols] = X[bool_cols].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=random_state)

scaler = StandardScaler()
X_train[non_bool_cols] = scaler.fit_transform(X_train[non_bool_cols])
X_test[non_bool_cols] = scaler.transform(X_test[non_bool_cols])

imputer = SimpleImputer(strategy='mean')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X.columns)

svm_smote = SMOTE(sampling_strategy=0.75, random_state=random_state)
X_res, y_res = svm_smote.fit_resample(X_train_imp, y_train)

majority_count = (y_res == 0).sum()
minority_count = (y_res == 1).sum()

catboost_model = models['CatBoost']()
lightgbm_model = models['LightGBM']()
ensemble_model = models['Ensemble']()

catboost_model.fit(X_res, y_res)
lightgbm_model.fit(X_res, y_res)
ensemble_model.fit(X_res, y_res)


y_pred_cat = catboost_model.predict(X_test_imp)
y_pred_lgb = lightgbm_model.predict(X_test_imp)
y_pred_ens = ensemble_model.predict(X_test_imp)

performance_results = []
fairness_results = []

for model_name, y_pred in [('CatBoost', y_pred_cat), ('LightGBM', y_pred_lgb), ('Ensemble', y_pred_ens)]:
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    
    performance_results.append({
        'Model': model_name,
        'Precision': precision,
        'Recall': recall
    })
    
    for attr in protected_attributes:
        sensitive_features_test = dataset.loc[X_test.index, attr]
        dp_diff = demographic_parity_difference(
            y_true=y_test,
            y_pred=y_pred,
            sensitive_features=sensitive_features_test
        )
    
        eo_diff = equalized_odds_difference(
            y_true=y_test,
            y_pred=y_pred,
            sensitive_features=sensitive_features_test
        )
        
        fairness_results.append({
            'Model': model_name, 
            'Attribute': attr,
            'Demographic Parity Diff': dp_diff,
            'Equalized Odds Diff': eo_diff
        })


performance_df = pd.DataFrame(performance_results)
fairness_df = pd.DataFrame(fairness_results)

long_perf_df = pd.melt(
    performance_df,
    id_vars=['Model'],
    value_vars=['Precision', 'Recall'],
    var_name='Metric',
    value_name='Score'
)

In [ ]:
fairness_long = fairness_df.melt(
    id_vars=['Model', 'Attribute'],
    value_vars=['Demographic Parity Diff', 'Equalized Odds Diff'],
    var_name='Fairness Metric',
    value_name='Score'
)

for metric in ['Demographic Parity Diff', 'Equalized Odds Diff']:
    plt.figure(figsize=(14, 7))
    sns.barplot(
        data=fairness_df,
        x='Attribute',
        y=metric,
        hue='Model',
        palette='Set2',
        ci=None,
        dodge=True
    )
    plt.title(f"{metric} Across Attributes")
    plt.axhline(0, linestyle='--', color='gray')
    plt.ylabel("Difference (Ideal = 0)")
    plt.xticks(rotation=45)
    plt.legend(title='Model')
    plt.tight_layout()
    plt.show()

for attr in protected_attributes:
    plt.figure(figsize=(14, 6))
    
    combined_data = []
    
    for model_name, y_pred in [('CatBoost', y_pred_cat),('LightGBM', y_pred_lgb), ('Ensemble', y_pred_ens)]:
        sensitive_features_test = df.loc[X_test.index, attr]
        
        mf = MetricFrame(
            metrics={'Precision': precision_score, 'Recall': recall_score},
            y_true=y_test,
            y_pred=y_pred,
            sensitive_features=sensitive_features_test
        )
        
        per_group_metrics = mf.by_group.reset_index()
        per_group_metrics['Model'] = model_name
        combined_data.append(per_group_metrics)
    
    combined_df = pd.concat(combined_data)
    
    combined_melted = combined_df.melt(
        id_vars=[attr, 'Model'],
        value_vars=['Precision', 'Recall'],
        var_name='Metric',
        value_name='Score'
    )
    
    combined_melted['Model_Metric'] = combined_melted['Model'] + ' | ' + combined_melted['Metric']
    
    sns.barplot(
        data=combined_melted,
        x=attr,
        y='Score',
        hue='Model_Metric',
        palette='Set2'
    )
    
    plt.title(f'Precision and Recall by Group for {attr}')
    plt.ylim(0, 1)
    plt.ylabel('Score')
    plt.xlabel(attr)
    plt.xticks(rotation=45)
    plt.legend(title='Model | Metric', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()



In [ ]:
fairness_df

In [ ]:
performance_df

#### Preliminary Analysis of Model Performance and Fairness with Protected Attributes

##### Predictive Performance
- The **CatBoost** model achieves the highest precision (0.80), while the **Ensemble** model leads in recall (0.83).
- **LightGBM** performs comparably, with balanced precision (0.79) and recall (0.81).
- Overall, the models demonstrate strong predictive ability when trained with protected attributes included.

##### Fairness Metrics
- **Sex_int:** All models show low demographic parity differences (~0.02–0.04) and low to moderate equalized odds differences (~0.03–0.09), indicating relatively fair treatment across sex groups.
- **Protected category:** Exhibits consistently high demographic parity differences (~0.22–0.23) but low equalized odds differences (~0.03–0.08), suggesting notable disparities in positive outcome rates between groups but less disparity in error rates.
- **Age Range_int:** Moderate demographic parity differences (~0.10–0.12) with more pronounced equalized odds differences for CatBoost (0.60) and LightGBM/Ensemble (0.40), pointing to some fairness concerns regarding age.
- **Italian Residence:** Shows moderate to high demographic parity differences (~0.09–0.12) and high equalized odds differences (~0.78–0.84), indicating potential geographic bias and error rate disparities.
- **European Residence:** Demographic parity differences vary widely—from very low (0.003 in LightGBM) to moderate (~0.10–0.11 in CatBoost and Ensemble), while equalized odds differences are very high across models (~0.78–0.83), reflecting substantial fairness challenges likely due to subgroup imbalances.

##### Next Steps
To mitigate these fairness concerns, we will experiment with **removing protected attributes** from the training data. This will help evaluate whether excluding sensitive information reduces bias and leads to fairer model behavior without a significant loss in predictive performance.


### Removing Protected Atttributes

In [ ]:
df = df_cleaned.copy()

protected_attributes = [
    'Sex_int', 'Protected category', 'Age Range_int',
    'Italian Residence', 'European Residence'
]

df = df.dropna(subset=protected_attributes).reset_index(drop=True)

feature_sets_dict = {
    'With Protected': feature_sets['custom_scores_with_essential_base_attributes'],
    'Without Protected': list(set(feature_sets['custom_scores_with_essential_base_attributes']) - set(protected_attributes))
}

results = []
pergroup_all = []

for setting_label, features in feature_sets_dict.items():
    X = df[features].copy()
    y = df['Hired']

    bool_cols = X.select_dtypes(include='bool').columns
    non_bool_cols = X.columns.difference(bool_cols)
    X[bool_cols] = X[bool_cols].astype(int)
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=random_state)
    scaler = StandardScaler()
    X_train[non_bool_cols] = scaler.fit_transform(X_train[non_bool_cols])
    X_test[non_bool_cols] = scaler.transform(X_test[non_bool_cols])

    imputer = SimpleImputer(strategy='mean')
    X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns)
    X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X.columns)

    X_res, y_res = SMOTE(sampling_strategy=0.5, random_state=random_state).fit_resample(X_train_imp, y_train)

    catboost_model = models['CatBoost']()
    lightgbm_model = models['LightGBM']()
    ensemble_model = models['Ensemble']()

    catboost_model.fit(X_res, y_res)
    lightgbm_model.fit(X_res, y_res)
    ensemble_model.fit(X_res, y_res)

    y_pred_cat = catboost_model.predict(X_test_imp)
    y_pred_lgb = lightgbm_model.predict(X_test_imp)
    y_pred_ens = ensemble_model.predict(X_test_imp)

    for model_name, y_pred in [('CatBoost', y_pred_cat), ('LightGBM', y_pred_lgb), ('Ensemble', y_pred_ens)]:
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        results.append({
            'Setting': setting_label,
            'Model': model_name,
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1
        })

        for attr in protected_attributes:
            sensitive_features_test = df.loc[X_test.index, attr]

            dp_diff = demographic_parity_difference(y_test, y_pred, sensitive_features=sensitive_features_test)
            eo_diff = equalized_odds_difference(y_test, y_pred, sensitive_features=sensitive_features_test)

            results.append({
                'Setting': setting_label,
                'Model': model_name,
                'Attribute': attr,
                'Metric': 'Demographic Parity Diff',
                'Score': dp_diff
            })
            results.append({
                'Setting': setting_label,
                'Model': model_name,
                'Attribute': attr,
                'Metric': 'Equalized Odds Diff',
                'Score': eo_diff
            })

            mf = MetricFrame(
                metrics={'precision': precision_score, 'recall': recall_score},
                y_true=y_test,
                y_pred=y_pred,
                sensitive_features=sensitive_features_test
            )

            pergroup_all.append(pd.DataFrame({
                'Setting': setting_label,
                'Model': model_name,
                'Attribute': attr,
                'Group': mf.by_group.index,
                'Precision': mf.by_group['precision'].values,
                'Recall': mf.by_group['recall'].values
            }))

performance_df = pd.DataFrame([r for r in results if 'F1 Score' in r])
fairness_df = pd.DataFrame([r for r in results if 'Attribute' in r])
pergroup_df = pd.concat(pergroup_all, ignore_index=True)


In [ ]:
performance_df

In [ ]:
fairness_df

In [ ]:
pergroup_df

In [ ]:
fairness_df['Model_Setting'] = fairness_df['Model'] + ' | ' + fairness_df['Setting']

plt.figure(figsize=(18, 7))

fairness_df['Attribute_Model'] = fairness_df['Attribute'] + ' | ' + fairness_df['Model']

metrics = fairness_df['Metric'].unique()

for metric in metrics:
    plt.figure(figsize=(18, 7))
    subset = fairness_df[fairness_df['Metric'] == metric]

    order = sorted(subset['Attribute_Model'].unique(), key=lambda x: x.split(' | ')[0])
    
    sns.barplot(
        data=subset,
        x='Attribute_Model',
        y='Score',
        hue='Setting',          
        palette='Set2',
        ci=None,
        order=order
    )
    
    plt.title(f"Fairness Metric: {metric} (With vs Without Protected Attributes)")
    plt.axhline(0, linestyle='--', color='gray')
    plt.xticks(rotation=75, ha='right')
    plt.ylabel('Score')
    plt.xlabel('Attribute | Model')
    plt.legend(title='Setting')
    plt.tight_layout()
    plt.show()


In [ ]:
performance_df['Model+Setting'] = performance_df['Model'] + ' | ' + performance_df['Setting']
fairness_df['Model+Setting'] = fairness_df['Model'] + ' | ' + fairness_df['Setting']
pergroup_df['Model+Setting'] = pergroup_df['Model'] + ' | ' + pergroup_df['Setting']

plt.figure(figsize=(10, 6))
melted_perf = performance_df.melt(
    id_vars=['Model+Setting'], 
    value_vars=['Precision', 'Recall', 'F1 Score'],
    var_name='Metric', 
    value_name='Value'
)
sns.barplot(data=melted_perf, x='Model+Setting', y='Value', hue='Metric', palette='pastel')
plt.title("Model Performance: With vs Without Protected Attributes")
plt.xticks(rotation=45)
plt.ylim(0.6, 1)
plt.tight_layout()
plt.show()



for attr in protected_attributes:
    df_attr = pergroup_df[pergroup_df['Attribute'] == attr]

    for metric in ['Precision', 'Recall']:
        plt.figure(figsize=(12, 6))
        sns.barplot(
            data=df_attr,
            x='Group',
            y=metric,
            hue='Model+Setting',
            palette='Set2'
        )
        plt.title(f"{metric} by Group for '{attr}': With vs Without Protected Attributes")
        plt.ylim(0, 1)
        plt.xticks(rotation=45)
        plt.legend(title='Model + Setting')
        plt.tight_layout()
        plt.show()




#### Analysis Summary: Impact of Including Protected Attributes on Model Performance and Fairness

##### Predictive Performance
Including protected attributes **consistently improves model performance** across all three models:

- **CatBoost** saw an increase in precision from **0.780 → 0.806**, and in F1 score from **0.804 → 0.823**.
- **LightGBM** maintained similar precision (**0.780**) but improved slightly in recall.
- **Ensemble** benefited the most, with precision rising to **0.784** and F1 score to **0.816**, making it the top performer in the "With Protected" setting.

Overall, incorporating protected features allows models to better capture patterns associated with subgroup characteristics, boosting predictive accuracy and balance.

---

##### Fairness Metrics

**Demographic Parity Difference (DPD)**

- There is **no systematic reduction or increase** in DPD when protected attributes are **omitted**, even though the goal of omission was to improve fairness.
- Changes vary significantly by model and group:
  - *CatBoost → Age Range*: DPD increases from **0.135 (Without)** to **0.168 (With)**.
  - *LightGBM → Protected category*: DPD stays consistently low (**0.013**) in both settings.

**Conclusion**: Although we expected that **omitting protected attributes would reduce bias**, the results show **no consistent improvement in demographic parity**. Impacts are **highly model- and attribute-dependent**.

**Equalized Odds Difference (EOD)**

- Similarly, **no clear trend emerges** when protected attributes are removed:
  - *CatBoost → Protected category*: EOD increases slightly from **0.839 (Without)** to **0.849 (With)**.
  - *Ensemble → European Residence*: EOD also increases from **0.819 (Without) → 0.851 (With)**.
  - *Ensemble → Sex_int*: EOD improves, decreasing from **0.058 (Without) → 0.017 (With)**.

**Conclusion**: Despite the intention that **removing protected attributes might improve equalized odds**, the data shows **no systematic benefit**. In some cases, disparities even worsen, reinforcing that **omission alone is not a reliable fairness strategy**.

---

##### Key Takeaways
- **Including protected attributes improves overall accuracy and helps mitigate demographic parity gaps.**
- **Equalized odds disparities persist**, especially for sensitive attributes like “Protected category” and “European Residence”.
- **Ensemble model** consistently outperforms others in balancing performance and fairness when protected attributes are used.
- **Omitting protected features** generally results in **slightly lower or unchanged F1 scores**, and **does not lead to consistent improvements in fairness metrics**. In some cases, fairness disparities even **increase**, indicating that omission is **not a reliable fairness strategy**.
